In [22]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(seed=12345)

In [21]:
#number of simulated years
years = 10000
#policies in portfolio
policies = 1000
#expected claims in a year per policy
lam = .1
#average cost of each claim
mean_severity = 5000
#coefficient of variation
CV = 1.5
#Shape 
shape = 1/(CV**2)
#Scale for gamma 
scale = mean_severity * (CV**2)

#checking shape and scale
print(f'Shape: {shape}\nScale: {scale}')


Shape: 0.4444444444444444
Scale: 11250.0


In [23]:
#use Poisson randomizer to simulate number of claims in one year
n_claims = rng.poisson(lam*policies)

#Use gamma distribution to find array of simulated damages from each claim
claim_amounts = rng.gamma(shape,scale,size=n_claims)

#Total of the claim damages from above
agg_loss = claim_amounts.sum()

print(f'Number of Claims: {n_claims}')
print('Example Claim Amounts:')
for amount in claim_amounts[:5]:
    print(f'   ${amount:,.2f}')
print(f'Total Loss: ${agg_loss:,.2f}')

Number of Claims: 91
Example Claim Amounts:
   $8,978.71
   $1,360.89
   $4,907.25
   $489.37
   $1,790.60
Total Loss: $427,737.18


It is important to note why Poisson was used in this simulation. Poisson allows us to evaluate 1,000 individual policies in a single year because of its properties. Adding two (or more) independent Poisson distributions creates a Poisson distribution with a lambda equal to the sum of the independent distributions’ lambdas. Hence, in a single year with 1,000 policies, each with a lambda of 0.1, the total claim count can be represented as a single Poisson distribution with a lambda of 100. This property requires all the policies to be independent. This could be a limiting factor in reality, as some policies might not be independent.

In [27]:
agg_losses = []
i = 0
#I am now going to loop through 10000 years of simulation using the process I used above. I will store each years loss in a list.
while i < years:
    n_claims = rng.poisson(lam*policies)

    claim_amounts = rng.gamma(shape,scale,size=n_claims)

    agg_loss = claim_amounts.sum()
    
    i=i+1
    agg_losses.append(agg_loss)
    
loss_array = np.array(agg_losses)

print(len(loss_array))
print(loss_array[:5])

10000
[468262.59226754 393510.19867716 386727.17922303 380747.12463301
 602499.20000762]


In [46]:
# Summary statistics of simulated annual aggregate losses
mean_loss = loss_array.mean()
std_loss = loss_array.std()
max_loss = loss_array.max()
min_loss = loss_array.min()

# Value at Risk (VaR) at the 90th, 95th, and 99th percentiles
var_90 = np.percentile(loss_array, 90)
var_95 = np.percentile(loss_array, 95)
var_99 = np.percentile(loss_array, 99)

print(f'Mean annual aggregate loss: ${mean_loss:,.2f}')
print(f'Standard deviation: ${std_loss:,.2f}')
print(f'Maximum annual loss: ${max_loss:,.2f}')
print(f'Minimum annual loss: ${min_loss:,.2f}\n')

print(f'90th percentile (VaR 90%): ${var_90:,.2f}')
print(f'95th percentile (VaR 95%): ${var_95:,.2f}')
print(f'99th percentile (VaR 99%): ${var_99:,.2f}')



Mean annual aggregate loss: $500,874.25
Standard deviation: $90,148.68
Maximum annual loss: $879,482.12
Minimum annual loss: $222,017.52

90th percentile (VaR 90%): $621,396.72
95th percentile (VaR 95%): $655,744.54
99th percentile (VaR 99%): $734,731.31


In 99% of simulated years, total annual losses stayed below $734,731. In other words, a loss year worse than this is expected roughly once every 100 years.

The mean of the simulation was 500,874 per year, which aligns closely with the theoretical expectation of 500,000.The 874 difference is within one standard error of the simulated mean, roughly 900, so it is consistent with normal simulation variation.

The difference between the maximum loss and the mean is 378,607, while the difference between the mean and the minimum is 278,856. This indicates mild right skew in the distribution of annual losses. The skew comes mainly from claim severity: the Gamma distribution has a long right tail, so occasional very large claims can push a year’s total well above average, while nothing comparable pulls a year far below it.

In [54]:
#Now we want to evaluate the years that are uniqely bad
badyears = loss_array[loss_array>=var_99]
tvar99 = badyears.mean()
print(f'The amount of years that has losses greater than the 99th percentile is {len(badyears)}')
print(f'In those years, the average loss is ${badyears.mean():,.2f}')



The amount of years that has losses greater than the 99th percentile is 100
In those years, the average loss is $773,828.58


In the worst 1% of simulated years, the average annual loss was 773,828, compared with a VaR 99% of 734,731. As an insurer, this value is just as important as the 99th percentile itself. VaR tells us where the tail begins, but it says nothing about how severe losses are beyond that point. TVaR captures the severity of the tail. Two portfolios can have identical VaR and very different tail risk, and only TVaR reveals the difference. This is one reason many risk frameworks have moved toward TVaR, also called Expected Shortfall. For example, international bank capital standards replaced VaR with Expected Shortfall for market risk.

In this simulation, TVaR is only about 5% above VaR, which indicates a relatively thin tail: once losses pass the 1-in-100 threshold, they don’t become dramatically more extreme. This is consistent with the mild right skew observed in Part 4. If TVaR were much larger than VaR, for example double, it would signal that rare years are extreme outliers that require significantly more capital to prepare for.

In [55]:
# Theoretical values for the compound Poisson-Gamma model
expected_claims = lam * policies                 
var_claims = expected_claims                     

var_severity = (CV * mean_severity) ** 2         

theory_mean = expected_claims * mean_severity    
theory_var = (expected_claims * var_severity
              + var_claims * mean_severity ** 2)  
theory_std = np.sqrt(theory_var)

# Compare simulated vs. theoretical
mean_diff = (mean_loss - theory_mean) / theory_mean * 100
std_diff = (std_loss - theory_std) / theory_std * 100

print(f'Theoretical mean: ${theory_mean:,.2f}')
print(f'Simulated mean:   ${mean_loss:,.2f}  ({mean_diff:.2f}% difference)\n')
print(f'Theoretical std:  ${theory_std:,.2f}')
print(f'Simulated std:    ${std_loss:,.2f}  ({std_diff:.2f}% difference)')

Theoretical mean: $500,000.00
Simulated mean:   $500,874.25  (0.17% difference)

Theoretical std:  $90,138.78
Simulated std:    $90,148.68  (0.01% difference)
